<a href="https://colab.research.google.com/github/hikari1215/wafer-defect-identification/blob/main/wafer_defect_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Oct 28 04:04:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!apt-get install unrar -y

!unrar x -o+ /content/drive/MyDrive/images.rar /content/


!unrar x -o+ /content/drive/MyDrive/images.rar /content/

Streaming output truncated to the last 5000 lines.
Extracting  /content/images/class_4/641329d064cd41c69446fa7ac5315e4d.jpg      90%  OK 
Extracting  /content/images/class_4/6589a6097f8642c1bf8747f7b1c5334d.jpg      90%  OK 
Extracting  /content/images/class_4/6597aba611a74275bfb445bee214db62.jpg      90%  OK 
Extracting  /content/images/class_4/65b9a2287c434fd1943b95ca9e950a07.jpg      90%  OK 
Extracting  /content/images/class_4/667193f2213d45758e7aaf471bad9bfe.jpg      90%  OK 
Extracting  /content/images/class_4/674a77db1c0848ccbb818f6f63ca948c.jpg      90%  OK 
Extracting  /content/images/class_4/682f70d142da471c88b940b2c4068553.jpg      90%  OK 
Extracting  /content/images/class_4/6830f9ceb59a485681c6f5392493edc9.jpg      90%  OK 
Extracting  /content/images/class_4/68c2f118564c4893a616b4f48172dce3.jpg      90%  OK 
Extracting  /content/images/class_4/6c3c891a09f04c218aea3e44cbdb6baa.jpg      90%

In [1]:
import os
import shutil
from sklearn.model_selection import train_test_split

def stratified_dataset_split(src_dir, output_dir, test_size=0.2, random_state=42):

    # 1. Reset/Tạo thư mục đích
    train_dir = os.path.join(output_dir, 'train')
    val_dir = os.path.join(output_dir, 'val')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    classes = [d for d in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, d))]
    print(f"🔍 Found {len(classes)} Class: {classes}\n")

    for cls in sorted(classes):
        cls_path = os.path.join(src_dir, cls)
        images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]

        if len(images) == 0:
            print(f"None image in '{cls}' ")
            continue


        train_imgs, val_imgs = train_test_split(
            images,
            test_size=test_size,
            random_state=random_state,
            shuffle=True
        )


        out_train_cls = os.path.join(train_dir, cls)
        out_val_cls = os.path.join(val_dir, cls)
        os.makedirs(out_train_cls, exist_ok=True)
        os.makedirs(out_val_cls, exist_ok=True)

        # 4. Copy ảnh sang thư mục mới
        for img in train_imgs:
            shutil.copy(os.path.join(cls_path, img), os.path.join(out_train_cls, img))
        for img in val_imgs:
            shutil.copy(os.path.join(cls_path, img), os.path.join(out_val_cls, img))

        print(f" [{cls}]: Sum {len(images)} Images ➔ Train: {len(train_imgs)} | Val: {len(val_imgs)}")

    print("\nDone.")

# --- Thực thi ---
if __name__ == "__main__":
    # Thay đường dẫn folder chứa 6 class gốc và folder muốn lưu
    RAW_DATA_DIR = "/content/images"
    DATASET_SPLIT_DIR = "/content/dataset_cls"

    stratified_dataset_split(
        src_dir=RAW_DATA_DIR,
        output_dir=DATASET_SPLIT_DIR,
        test_size=0.2
    )

FileNotFoundError: [Errno 2] No such file or directory: '/content/images'

In [ ]:
!pip install ultralytics


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s-cls.pt")

In [ ]:
results = model.train(data="/content/dataset_cls",
                      epochs=50,
                      imgsz=480,
                      batch=32,
                     )  # run training (only run when the database is full)

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_cls, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-6, nbs=64, nms=None, opset=None, optimize=False, optimizer=

In [ ]:
import shutil

drive_destination = "/content/drive/MyDrive/Wafer Defect Identification"


shutil.copytree("/content/runs/classify/train-6", drive_destination, dirs_exist_ok=True)

print("done")

done
